# Notebook 2: Feature Engineering
**PURPOSE:** Convert raw ball-by-ball data into ML-ready features without data leakage.
Every feature must be known AT prediction time.

In [1]:
import os

# Fix working directory
project_root = r"C:\Users\srich\OneDrive\Desktop\prediction-model"
os.chdir(project_root)

# Verify correct directory
print(f"✅ Working directory set to:")
print(f"   {os.getcwd()}")

# Verify data files exist
data_files = [
    'data/raw/ipl_matches.csv',
    'data/raw/ipl_deliveries.csv',
    'data/raw/t20wc_matches.csv',
    'data/raw/t20wc_deliveries.csv',
    'data/raw/bbl_matches.csv',
    'data/raw/bbl_deliveries.csv'
]

print(f"\n📁 Data file check:")
all_good = True
for f in data_files:
    exists = os.path.exists(f)
    status = '✅' if exists else '❌'
    print(f"   {status} {f}")
    if not exists:
        all_good = False

if all_good:
    print("\n✅ All data files found!")
    print("   Ready to run notebook.")
else:
    print("\n❌ Some files missing!")
    print("   Check data/raw/ folder")
    raise FileNotFoundError(
        "Data files missing. "
        "Check data/raw/ folder."
    )

✅ Working directory set to:
   C:\Users\srich\OneDrive\Desktop\prediction-model

📁 Data file check:
   ✅ data/raw/ipl_matches.csv
   ✅ data/raw/ipl_deliveries.csv
   ✅ data/raw/t20wc_matches.csv
   ✅ data/raw/t20wc_deliveries.csv
   ✅ data/raw/bbl_matches.csv
   ✅ data/raw/bbl_deliveries.csv

✅ All data files found!
   Ready to run notebook.


In [2]:
import pandas as pd
import numpy as np
import json, os, warnings, pickle
from collections import deque
warnings.filterwarnings('ignore')
print('Libraries loaded ✅')

Libraries loaded ✅


## 2.1 Load Raw Data

In [3]:
ipl_matches = pd.read_csv('data/raw/ipl_matches.csv')
ipl_deliveries = pd.read_csv('data/raw/ipl_deliveries.csv', low_memory=False)
t20_matches = pd.read_csv('data/raw/t20wc_matches.csv')
t20_deliveries = pd.read_csv('data/raw/t20wc_deliveries.csv', low_memory=False)
bbl_matches = pd.read_csv('data/raw/bbl_matches.csv')
bbl_deliveries = pd.read_csv('data/raw/bbl_deliveries.csv', low_memory=False)

ipl_matches['league'] = 'IPL'
ipl_deliveries['league'] = 'IPL'
print('Data loaded ✅')
print(f'IPL deliveries columns: {list(ipl_deliveries.columns)}')

Data loaded ✅
IPL deliveries columns: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder', 'league']


## 2.2 Data Cleaning

In [4]:
def clean_matches(matches_df, league):
    df = matches_df.copy()
    name_map = {
        'Delhi Daredevils': 'Delhi Capitals',
        'Kings XI Punjab': 'Punjab Kings',
        'Rising Pune Supergiant': 'Rising Pune Supergiants',
        'Deccan Chargers': 'Sunrisers Hyderabad',
    }
    for col in ['team1', 'team2', 'toss_winner', 'winner']:
        if col in df.columns:
            df[col] = df[col].replace(name_map)
    before = len(df)
    df = df[df['winner'].notna()]
    df = df[df['result'] != 'no result']
    if 'method' in df.columns:
        df = df[df['method'].isna()]
    after = len(df)
    print(f'{league}: {before} → {after} matches after cleaning (removed {before-after})')
    return df

ipl_matches_clean = clean_matches(ipl_matches, 'IPL')
t20_matches_clean = clean_matches(t20_matches, 'T20I')
bbl_matches_clean = clean_matches(bbl_matches, 'BBL')

IPL: 1095 → 1069 matches after cleaning (removed 26)
T20I: 3111 → 3002 matches after cleaning (removed 109)
BBL: 647 → 615 matches after cleaning (removed 32)


## 2.3 Feature Engineering Function
Processes each match ball-by-ball, maintaining running state.
**CRITICAL:** No future information is used — only data available at prediction time.

In [5]:
def engineer_match_features(match_deliveries, match_info, col_map):
    """Process a single match into ML-ready feature rows.
    
    Args:
        match_deliveries: DataFrame of ball-by-ball data for one match
        match_info: dict with match metadata (id, winner, city, etc.)
        col_map: dict mapping logical column names to actual column names
    """
    rows = []
    match_id = match_info['id']
    winner = match_info.get('winner')
    city = str(match_info.get('city', 'Unknown'))
    if pd.isna(city) or city == 'nan':
        venue = str(match_info.get('venue', 'Unknown'))
        city = venue.split(',')[0].strip()
    league = match_info.get('league', col_map.get('league', 'IPL'))

    # Unpack col_map
    inning_col = col_map['inning_col']
    runs_col = col_map['runs_col']
    bat_col = col_map['bat_col']
    ball_combined = col_map.get('ball_combined', False)
    over_col = col_map.get('over_col')
    ball_col = col_map.get('ball_col', 'ball')
    extras_type_col = col_map.get('extras_type_col')
    wide_col_name = col_map.get('wide_col')
    nb_col_name = col_map.get('nb_col')
    bye_col_name = col_map.get('bye_col')
    lb_col_name = col_map.get('lb_col')
    wicket_col = col_map.get('wicket_col', 'player_dismissed')

    # --- Filter innings (cast to int to handle string/float types) ---
    inn2 = match_deliveries[match_deliveries[inning_col].astype(int) == 2].copy()
    if len(inn2) == 0:
        return rows

    batting_team_inn2 = inn2['batting_team'].iloc[0]
    bowling_team_inn2 = inn2['bowling_team'].iloc[0]

    inn1 = match_deliveries[match_deliveries[inning_col].astype(int) == 1].copy()
    if len(inn1) == 0:
        return rows

    inn1_runs = inn1[runs_col].sum()
    target = int(inn1_runs) + 1

    # --- Running state ---
    runs_scored = 0; wickets = 0; extras = 0; wides = 0
    no_balls_count = 0; byes = 0; leg_byes = 0
    boundaries_4 = 0; boundaries_6 = 0
    dot_balls = 0; legitimate_balls = 0
    last_12_balls = deque(maxlen=12)
    over_runs = {}; current_over_runs = 0; last_over = -1
    partnership_runs = 0; partnership_balls = 0

    for _, ball_row in inn2.iterrows():
        try:
            # ── Extract over & ball_num ──────────────────────
            if ball_combined:
                # T20I/BBL: ball column is float like 0.1, 1.5, 2.10
                val = float(ball_row[ball_col])
                over = int(val)
                ball_num = int(round((val % 1) * 10))
                ball_num = min(ball_num, 6)
            else:
                # IPL: separate over and ball columns
                over = int(ball_row.get(over_col, 0))
                ball_num = int(ball_row.get(ball_col, 1))

            # ── Track over transitions ──────────────────────
            if over != last_over:
                if last_over >= 0:
                    over_runs[last_over] = current_over_runs
                current_over_runs = 0
                last_over = over

            # ── Extract runs ────────────────────────────────
            def _safe_float(v):
                try:
                    v2 = float(v)
                    return 0.0 if v2 != v2 else v2
                except (TypeError, ValueError):
                    return 0.0
            runs_bat = _safe_float(ball_row.get(bat_col, 0))
            run_total = _safe_float(ball_row.get(runs_col, 0))

            # ── Parse extras (two formats) ──────────────────
            if extras_type_col:
                # IPL format: extras_type is a string column
                extras_type_val = str(
                    ball_row.get(extras_type_col, '')
                ).lower()
                _erv = ball_row.get('extra_runs', 0)
                try:
                    extra_runs_val = float(_erv)
                    if extra_runs_val != extra_runs_val:
                        extra_runs_val = 0.0
                except (TypeError, ValueError):
                    extra_runs_val = 0.0
                is_wide = extras_type_val == 'wides'
                is_nb = extras_type_val == 'noballs'
                is_bye = extras_type_val == 'byes'
                is_lb = extras_type_val == 'legbyes'
                wide = extra_runs_val if is_wide else 0.0
                nb_val = extra_runs_val if is_nb else 0.0
                bye = extra_runs_val if is_bye else 0.0
                lb = extra_runs_val if is_lb else 0.0
                runs_ext = extra_runs_val
            else:
                # T20I/BBL format: numeric columns
                # NaN-safe: extras cols are NaN (not 0)
                # when no extra on the delivery
                def _safe(v):
                    try:
                        v2 = float(v)
                        return 0.0 if v2 != v2 else v2
                    except (TypeError, ValueError):
                        return 0.0
                wide = _safe(
                    ball_row.get(wide_col_name, 0)
                ) if wide_col_name else 0.0
                nb_val = _safe(
                    ball_row.get(nb_col_name, 0)
                ) if nb_col_name else 0.0
                bye = _safe(
                    ball_row.get(bye_col_name, 0)
                ) if bye_col_name else 0.0
                lb = _safe(
                    ball_row.get(lb_col_name, 0)
                ) if lb_col_name else 0.0
                is_wide = wide > 0
                is_nb = nb_val > 0
                runs_ext = _safe(
                    ball_row.get('runs_extras', 0)
                )

            # ── Accumulate totals ───────────────────────────
            runs_scored += run_total
            extras += runs_ext
            wides += wide
            no_balls_count += nb_val
            byes += bye
            leg_byes += lb
            current_over_runs += run_total

            # ── Legitimate ball detection ───────────────────
            # Wides are NOT legitimate; no-balls ARE legitimate
            if not is_wide:
                legitimate_balls += 1
                last_12_balls.append(runs_bat)
                partnership_balls += 1
                if runs_bat == 0:
                    dot_balls += 1
                elif runs_bat == 4:
                    boundaries_4 += 1
                elif runs_bat == 6:
                    boundaries_6 += 1

            # ── Wicket detection (both formats) ─────────────
            if wicket_col == 'is_wicket':
                is_wkt = bool(ball_row.get('is_wicket', False))
            else:
                dismissed = ball_row.get(
                    'player_dismissed', None
                )
                is_wkt = (
                    dismissed is not None
                    and str(dismissed) not in
                        ['nan', '', 'None']
                )

            if is_wkt:
                wickets += 1
                partnership_runs = 0
                partnership_balls = 0
            else:
                partnership_runs += run_total

            # ── Compute match-state features ────────────────
            balls_faced = (over * 6) + ball_num
            overs_done = balls_faced / 6
            balls_left = max(0, 120 - balls_faced)
            if balls_left <= 0:
                continue

            runs_left = max(0, target - runs_scored)
            wickets_remaining = 10 - wickets
            crr = (runs_scored / overs_done
                   if overs_done > 0 else 0)
            rrr = (runs_left / (balls_left / 6)
                   if balls_left > 0 else 99.99)

            recent_overs = list(over_runs.values())[-3:]
            last_3_avg = (np.mean(recent_overs)
                          if recent_overs else crr)
            recent_12_rr = (sum(last_12_balls) / 2.0
                            if len(last_12_balls) > 0
                            else crr)

            row = {
                'runs_left': runs_left,
                'balls_left': balls_left,
                'wickets_remaining': wickets_remaining,
                'crr': round(crr, 3),
                'rrr': round(rrr, 3),
                'run_rate_diff': round(crr - rrr, 3),
                'batting_team': batting_team_inn2,
                'bowling_team': bowling_team_inn2,
                'city': city,
                'league': league,
                'over_number': over + 1,
                'is_powerplay': int(over < 6),
                'is_middle_overs': int(6 <= over < 15),
                'is_death_overs': int(over >= 15),
                'pressure_index': round(
                    rrr / max(crr, 0.1), 3),
                'required_balls_per_run': round(
                    balls_left / max(runs_left, 1), 3),
                'wickets_pressure': round(
                    wickets / max(over + 1, 1), 3),
                'recent_12_balls_rr': round(
                    recent_12_rr, 3),
                'last_3_overs_avg': round(
                    last_3_avg, 3),
                'momentum_vs_required': round(
                    recent_12_rr - rrr, 3),
                'total_extras': extras,
                'extras_rate': round(
                    extras / max(over + 1, 1), 3),
                'wides_this_innings': wides,
                'extra_balls_gained': wides + no_balls_count,
                'boundary_percentage': round(
                    (boundaries_4 + boundaries_6)
                    / max(legitimate_balls, 1), 3),
                'dot_ball_percentage': round(
                    dot_balls
                    / max(legitimate_balls, 1), 3),
                'partnership_runs': partnership_runs,
                'partnership_balls': partnership_balls,
                'result': (int(winner == batting_team_inn2)
                           if winner else None),
            }
            if over >= 4 and row['result'] is not None:
                rows.append(row)
        except Exception:
            continue
    return rows

print('Feature engineering function defined ✅')

Feature engineering function defined ✅


## 2.4 Process All Matches
**Note:** This takes ~5-10 minutes.

In [6]:
def process_all_matches(deliveries_df, matches_df, league_name):
    """Process all matches for a league with dynamic column detection."""
    all_rows = []
    cols = list(deliveries_df.columns)
    print(f'\n{"="*50}')
    print(f'{league_name} COLUMN DETECTION')
    print(f'{"="*50}')
    print(f'Available columns: {cols}')

    # ── Step 1: Detect inning column ────────────────────
    inning_col = next(
        (c for c in ['inning', 'innings', 'Inning', 'Innings']
         if c in cols), None
    )
    if inning_col is None:
        print(f'❌ FATAL: No inning column in {league_name}')
        return all_rows
    inning_values = deliveries_df[inning_col].unique()
    print(f'Inning column: {inning_col} → values: {inning_values}')

    # ── Step 2: Detect runs columns ─────────────────────
    runs_col = next(
        (c for c in ['total_runs', 'runs_total']
         if c in cols), None
    )
    if runs_col is None:
        print(f'❌ FATAL: No runs column in {league_name}')
        return all_rows

    bat_col = next(
        (c for c in ['batsman_runs', 'runs_batsman',
                     'runs_off_bat']
         if c in cols), None
    )
    if bat_col is None:
        print(f'❌ FATAL: No batsman runs column in {league_name}')
        return all_rows

    # ── Step 3: Detect over/ball format ─────────────────
    has_over_col = 'over' in cols
    ball_combined = not has_over_col
    over_col = 'over' if has_over_col else None

    # ── Step 4: Detect extras columns ───────────────────
    wide_col = next(
        (c for c in ['wide_runs', 'wides'] if c in cols),
        None
    )
    nb_col = next(
        (c for c in ['noball_runs', 'noballs'] if c in cols),
        None
    )
    bye_col = next(
        (c for c in ['bye_runs', 'byes'] if c in cols),
        None
    )
    lb_col = next(
        (c for c in ['legbye_runs', 'legbyes'] if c in cols),
        None
    )
    extras_type_col = (
        'extras_type' if 'extras_type' in cols else None
    )

    # ── Step 5: Detect match_id column ──────────────────
    match_id_col = next(
        (c for c in ['match_id', 'id', 'match_no']
         if c in cols), None
    )
    if match_id_col is None:
        print(f'❌ FATAL: No match_id column in {league_name}')
        return all_rows

    # ── Step 6: Detect wicket column ────────────────────
    if 'is_wicket' in cols:
        wicket_col = 'is_wicket'
    else:
        wicket_col = 'player_dismissed'

    # ── Step 7: Verify match_id linkage ─────────────────
    delivery_ids = set(deliveries_df[match_id_col].unique())
    match_ids_set = set(matches_df['id'].unique())
    overlap = delivery_ids & match_ids_set
    print(f'Match ID linkage: {len(overlap)} overlap '
          f'out of {len(match_ids_set)} matches')
    if len(overlap) == 0:
        print(f'❌ FATAL: Zero match_id overlap in '
              f'{league_name}!')
        sample_d = list(delivery_ids)[:3]
        sample_m = list(match_ids_set)[:3]
        print(f'  Delivery IDs sample: {sample_d}')
        print(f'  Match IDs sample: {sample_m}')
        return all_rows

    # ── Build col_map ───────────────────────────────────
    col_map = {
        'inning_col': inning_col,
        'runs_col': runs_col,
        'bat_col': bat_col,
        'over_col': over_col,
        'ball_col': 'ball',
        'ball_combined': ball_combined,
        'wide_col': wide_col,
        'nb_col': nb_col,
        'bye_col': bye_col,
        'lb_col': lb_col,
        'extras_type_col': extras_type_col,
        'match_id_col': match_id_col,
        'wicket_col': wicket_col,
        'league': league_name,
    }

    # ── Print confirmed mapping ─────────────────────────
    print(f'\n{league_name} column mapping confirmed:')
    if ball_combined:
        print(f'  ball_combined=True '
              f'(combined over.ball format)')
    else:
        print(f'  ball_combined=False, '
              f'over={over_col}, ball=ball')
    if extras_type_col:
        print(f'  extras_type_col={extras_type_col} '
              f'(IPL mode)')
    print(f'  inning={inning_col}, runs={runs_col}, '
          f'bat={bat_col}')
    print(f'  wide={wide_col}, nb={nb_col}, '
          f'bye={bye_col}, lb={lb_col}')
    print(f'  match_id={match_id_col}, '
          f'wicket={wicket_col}')

    # ── Process each match ──────────────────────────────
    match_ids = matches_df['id'].unique()
    errors = 0
    print(f'  Processing {len(match_ids)} matches...')

    for i, match_id in enumerate(match_ids):
        if (i + 1) % 500 == 0:
            print(f'  Progress: {i+1}/{len(match_ids)}')
        try:
            match_dels = deliveries_df[
                deliveries_df[match_id_col] == match_id
            ]
            if len(match_dels) == 0:
                continue
            match_info_row = matches_df[
                matches_df['id'] == match_id
            ].iloc[0]
            match_info = match_info_row.to_dict()
            feat_rows = engineer_match_features(
                match_dels, match_info, col_map
            )
            all_rows.extend(feat_rows)
        except Exception:
            errors += 1
            continue

    print(f'\n{league_name} complete: '
          f'{len(all_rows):,} rows, {errors} errors')

    # ── Debug output if 0 rows ──────────────────────────
    if len(all_rows) == 0:
        print(f'\n⚠️  DEBUG: {league_name} produced 0 rows')
        print(f'col_map: {col_map}')
        first_mid = match_ids[0]
        print(f'First match_id: {first_mid}')
        sample = deliveries_df[
            deliveries_df[match_id_col] == first_mid
        ]
        print(f'Deliveries for first match: '
              f'{len(sample)} rows')
        if len(sample) > 0:
            print(sample.head(3).to_string())

    return all_rows


print('Starting feature engineering...')
ipl_rows = process_all_matches(
    ipl_deliveries, ipl_matches_clean, 'IPL')
t20_rows = process_all_matches(
    t20_deliveries, t20_matches_clean, 'T20I')
bbl_rows = process_all_matches(
    bbl_deliveries, bbl_matches_clean, 'BBL')

all_rows = ipl_rows + t20_rows + bbl_rows
df = pd.DataFrame(all_rows)

print(f'\n{"="*50}')
print(f'COMBINED DATASET')
print(f'{"="*50}')
print(f'Total rows:    {len(df):,}')
print(f'Total columns: {len(df.columns)}')
print(f'IPL rows:      {len(ipl_rows):,}')
print(f'T20I rows:     {len(t20_rows):,}')
print(f'BBL rows:      {len(bbl_rows):,}')
print(f'League values: {df["league"].nunique()} '
      f'({list(df["league"].unique())})')

Starting feature engineering...

IPL COLUMN DETECTION
Available columns: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder', 'league']
Inning column: inning → values: [1 2 3 4 5 6]
Match ID linkage: 1069 overlap out of 1069 matches

IPL column mapping confirmed:
  ball_combined=False, over=over, ball=ball
  extras_type_col=extras_type (IPL mode)
  inning=inning, runs=total_runs, bat=batsman_runs
  wide=None, nb=None, bye=None, lb=None
  match_id=match_id, wicket=is_wicket
  Processing 1069 matches...
  Progress: 500/1069
  Progress: 1000/1069

IPL complete: 97,048 rows, 0 errors

T20I COLUMN DETECTION
Available columns: ['match_id', 'season', 'date', 'venue', 'inning', 'ball', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_batsman', 'runs_extras', 'wide_runs', 'noball_runs', 'bye_runs', 

## 2.5 Data Validation

In [7]:
print('=== VALIDATION CHECKS ===\n')
nan_counts = df.isnull().sum()
print('NaN counts per feature:')
print(nan_counts[nan_counts > 0])
if nan_counts.sum() == 0:
    print('✅ Zero NaN values!')

balance = df['result'].value_counts(normalize=True)
print(f'\nClass balance:')
print(f'  Chasing wins:  {balance.get(1, 0):.1%}')
print(f'  Chasing loses: {balance.get(0, 0):.1%}')
if 0.40 <= balance.get(1, 0) <= 0.60:
    print('✅ Good class balance!')
else:
    print('⚠️  Imbalanced classes')

print('\nFeature statistics:')
print(df.describe().round(2))

=== VALIDATION CHECKS ===

NaN counts per feature:
Series([], dtype: int64)
✅ Zero NaN values!

Class balance:
  Chasing wins:  45.3%
  Chasing loses: 54.7%
✅ Good class balance!

Feature statistics:
       runs_left  balls_left  wickets_remaining        crr        rrr  \
count  398594.00   398594.00          398594.00  398594.00  398594.00   
mean       78.10       52.10               6.68       7.36      11.60   
std        44.05       26.12               2.21       1.93      19.31   
min         0.00        1.00               0.00       0.93       0.00   
25%        44.00       30.00               5.00       6.10       6.57   
50%        76.00       53.00               7.00       7.31       9.00   
75%       109.00       75.00               8.00       8.54      11.92   
max       328.00       95.00              10.00      21.36    1308.00   

       run_rate_diff  over_number  is_powerplay  is_middle_overs  \
count      398594.00    398594.00     398594.00        398594.00   
mean  

## 2.6 Train/Test Split — Temporal
**CRITICAL:** Temporal split prevents data leakage. Train on older data, test on newer.

In [8]:
ipl_df = df[df['league'] == 'IPL'].copy()
non_ipl_df = df[df['league'] != 'IPL'].copy()

ipl_train_size = int(len(ipl_df) * 0.80)
ipl_train = ipl_df.iloc[:ipl_train_size]
ipl_test = ipl_df.iloc[ipl_train_size:]

train_df = pd.concat([ipl_train, non_ipl_df], ignore_index=True)
test_df = ipl_test.copy()

print(f'Training set: {len(train_df):,} rows')
print(f'Test set:     {len(test_df):,} rows')
print(f'Test %:       {len(test_df)/len(df):.1%}')
print(f'\nTrain class balance: {train_df["result"].mean():.1%}')
print(f'Test class balance:  {test_df["result"].mean():.1%}')

Training set: 379,184 rows
Test set:     19,410 rows
Test %:       4.9%

Train class balance: 45.3%
Test class balance:  45.0%


## 2.7 Encode Categorical Features

In [9]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['batting_team', 'bowling_team', 'city', 'league']
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    le.fit(df[col].astype(str))
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))
    label_encoders[col] = le
    print(f'{col}: {len(le.classes_)} unique values')

os.makedirs('model', exist_ok=True)
with open('model/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
print('\n✅ Saved: model/label_encoders.pkl')

batting_team: 131 unique values
bowling_team: 131 unique values
city: 220 unique values
league: 3 unique values

✅ Saved: model/label_encoders.pkl


## 2.8 Save Processed Data

In [10]:
feature_cols = [c for c in df.columns if c != 'result']

X_train = train_df[feature_cols]
y_train = train_df['result']
X_test = test_df[feature_cols]
y_test = test_df['result']

train_df.to_csv('data/processed/training_data.csv', index=False)
test_df.to_csv('data/processed/test_data.csv', index=False)

with open('model/feature_names.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)

print(f'✅ Saved training data')
print(f'✅ Saved test data')
print(f'✅ Saved feature names')
print(f'\nFeatures ({len(feature_cols)}):')
for i, feat in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {feat}')
print(f'\nX_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')

✅ Saved training data
✅ Saved test data
✅ Saved feature names

Features (28):
   1. runs_left
   2. balls_left
   3. wickets_remaining
   4. crr
   5. rrr
   6. run_rate_diff
   7. batting_team
   8. bowling_team
   9. city
  10. league
  11. over_number
  12. is_powerplay
  13. is_middle_overs
  14. is_death_overs
  15. pressure_index
  16. required_balls_per_run
  17. wickets_pressure
  18. recent_12_balls_rr
  19. last_3_overs_avg
  20. momentum_vs_required
  21. total_extras
  22. extras_rate
  23. wides_this_innings
  24. extra_balls_gained
  25. boundary_percentage
  26. dot_ball_percentage
  27. partnership_runs
  28. partnership_balls

X_train: (379184, 28)
X_test:  (19410, 28)
